# Feedforward → Direct Mesh

Demonstrates meshing directly from a MapAnything pointcloud without training a Gaussian splat. The reconstruction is loaded from the zarr cache written by `feedforward_methods.ipynb` — run that notebook first.

**Pipeline:** zarr cache → `FeedforwardResult.load_zarr()` → `pointcloud_to_mesh()` → mesh

`pointcloud_to_mesh` dispatches to any registered mesh method via the same interface. Only `open3d_tsdf` is implemented today; Poisson variants raise `NotImplementedError`.

Compare with the splat-based path in `docs/splats/create_mesh.ipynb` which goes through nerfstudio training.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import open3d as o3d

from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.mesh import pointcloud_to_mesh

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import zarr
import pyvista as pv

from collab_splats.mesh.utils import transfer_features_to_mesh
from collab_splats.pointcloud.utils import lift_features
from collab_splats.semantics.features import BaseFeatureExtractor

## §1 — Load cached reconstruction

Read the MapAnything result written by `feedforward_methods.ipynb`. The zarr store lives in `docs/source/.cache/` and is gitignored; run the prior notebook if it is absent.

In [ ]:
%run ../../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD  = "mapanything"

RECON     = CACHE_DIR / METHOD

# Load from zarr written by feedforward_methods.ipynb
_recon_cache = RECON / "reconstruction.zarr"
assert _recon_cache.exists(), (
    f"No reconstruction cache at {_recon_cache}. "
    "Run 02_pointcloud/feedforward_methods first."
)

result = FeedforwardResult.load_zarr(_recon_cache, load_images=True)
print(f"Loaded MapAnything reconstruction: {result.points.shape[0]:,} pts, {result.extrinsics.shape[0]} frames")

## §2 — TSDF meshing

Run TSDF fusion directly on the feedforward depth maps. `voxel_size=0.02` (2 cm) is coarser than the splat-derived path because feedforward depth maps are noisier than splat-rendered depths.

In [4]:
MESH_DIR = Path("/tmp/feedforward_mesh_demo/mesh")

# Mesh directly from the FeedforwardResult
mesh_result = pointcloud_to_mesh(
    result,
    MESH_DIR,
    method="open3d_tsdf",
    voxel_size=0.02,
    sdf_trunc=0.08,
    depth_trunc=10.0,
    clean_repair=False,  # meshlib clean/repair disabled (addPartByMask unavailable in this env)
)
print(f"Mesh saved → {mesh_result.mesh_path}")

TSDF integration:   0%|          | 0/80 [00:00<?, ?it/s]

TSDF integration:   1%|▏         | 1/80 [00:00<01:04,  1.22it/s]

TSDF integration:   2%|▎         | 2/80 [00:01<00:36,  2.16it/s]

TSDF integration:   4%|▍         | 3/80 [00:01<00:29,  2.60it/s]

TSDF integration:   5%|▌         | 4/80 [00:01<00:23,  3.20it/s]

...

TSDF integration:  98%|█████████▊| 78/80 [00:41<00:02,  1.22s/it]

TSDF integration:  99%|█████████▉| 79/80 [00:42<00:01,  1.12s/it]

TSDF integration: 100%|██████████| 80/80 [00:43<00:00,  1.03s/it]

TSDF integration: 100%|██████████| 80/80 [00:43<00:00,  1.85it/s]

Mesh saved → /tmp/feedforward_mesh_demo/mesh/mesh_tsdf.ply


## §3 — Inspect mesh

Load the saved mesh with open3d and report basic statistics.

In [5]:
# Inspect mesh statistics
mesh = o3d.io.read_triangle_mesh(str(mesh_result.mesh_path))
print(f"Vertices: {len(mesh.vertices):,}  Triangles: {len(mesh.triangles):,}")

Vertices: 2,752,669  Triangles: 4,712,371


## §4 — Feature lifting

Extract DINOv2 features for each frame, then lift to per-point 3D features using multi-view
confidence-weighted projection. `extract_and_cache` skips extraction on re-run if the cache
is valid.

In [ ]:
# Extract per-frame DINOv2 features — cached under MESH_DIR
FEATURE_CACHE_DIR = MESH_DIR / "features_cache"
extractor = BaseFeatureExtractor.get("dinov2")()
zarr_path = extractor.extract_and_cache(result.image_paths, FEATURE_CACHE_DIR)

# Load feature maps: list of (D, H_p, W_p) tensors
store = zarr.open(str(zarr_path), mode="r")
feature_maps = [
    torch.from_numpy(np.array(store["features"][i]))
    for i in range(store["features"].shape[0])
]

# Lift 2D features → (P, D) per-point features
lifted = lift_features(feature_maps, result)
result.features = lifted.numpy()

print(f"Lifted features shape: {result.features.shape}")  # (P, D)

## §5 — Transfer features to mesh vertices

Map per-point features to mesh vertices via KNN Gaussian-weighted aggregation.
Points farther than `sdf_trunc` from any vertex are excluded.

In [ ]:
# Load the mesh written by §2
mesh = o3d.io.read_triangle_mesh(str(mesh_result.mesh_path))
mesh.compute_vertex_normals()

# Transfer: (P, D) pointcloud features → (M, D) vertex features
vertex_features = transfer_features_to_mesh(result, mesh)

print(f"Mesh vertices:   {np.asarray(mesh.vertices).shape[0]}")
print(f"Vertex features: {vertex_features.shape}")   # (M, D)
print(f"Non-zero verts:  {(vertex_features.any(axis=1)).sum()}")

## §6 — Similarity query: pointcloud vs mesh

Same query vector runs against both `result.features` (pointcloud) and `vertex_features` (mesh).
Cosine similarity scores are normalized to [0, 1] for visualization.

In [ ]:
pv.set_jupyter_backend("html")

# --- Pointcloud similarity ---
pc_feats = torch.from_numpy(result.features)           # (P, D)
pc_feats_norm = F.normalize(pc_feats, dim=-1)

# --- Mesh vertex similarity ---
vx_feats = torch.from_numpy(vertex_features)           # (M, D)
vx_feats_norm = F.normalize(vx_feats, dim=-1)

# Use first point feature as query vector for meaningful spatial variation
# To use Talk2DINO text encoding:
#   text_extractor = BaseFeatureExtractor.get("talk2dino")()
#   query_vec = text_extractor.encode_text(["floor"])[0]  # (D,)
query_vec = pc_feats_norm[0]
query_norm = F.normalize(query_vec.unsqueeze(0), dim=-1)  # (1, D)

pc_sim = (pc_feats_norm @ query_norm.T).squeeze(-1).numpy()    # (P,)
vx_sim = (vx_feats_norm @ query_norm.T).squeeze(-1).numpy()   # (M,)

# Normalize to [0, 1]
def norm01(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-8)

pc_sim_n = norm01(pc_sim)
vx_sim_n = norm01(vx_sim)

print(f"Pointcloud similarity — mean: {pc_sim_n.mean():.3f}  max: {pc_sim_n.max():.3f}")
print(f"Mesh vertex similarity — mean: {vx_sim_n.mean():.3f}  max: {vx_sim_n.max():.3f}")

# Visualize mesh colored by similarity
mesh_pv = pv.read(str(mesh_result.mesh_path))
mesh_pv["similarity"] = vx_sim_n

pl = pv.Plotter()
pl.add_mesh(mesh_pv, scalars="similarity", cmap="hot", clim=[0, 1])
pl.show()